# Paired Drug Response Analysis

*A standalone Wizards-Staff workflow for paired before/after experiments.*

This notebook walks through computing fold/percent changes between two **paired** Lizard-Wizard runs — for example, the same wells imaged before and after a drug application. If your design is **unpaired** (independent treatment vs. control), use the main [`tutorial-notebook.ipynb`](tutorial-notebook.ipynb) instead.

## Prerequisites

- You've worked through Part 1 (setup) and Part 2 (unpaired comparisons) of the [main tutorial](tutorial-notebook.ipynb), or are otherwise comfortable instantiating an `Orb` and reading metadata.
- You have **two** Lizard-Wizard output folders to pair (Path A below), **or** you want to see the workflow on synthetic data derived from a single analyzed `Orb` (Path B).

## What you'll do

1. Load (or fabricate) two `Orb`s — one baseline, one dosing.
2. Run the standard analysis on both timepoints.
3. Pair samples across the two runs by a shared identifier (default: `Well`).
4. Compute fold change / percent change / delta and visualize the drug response.
5. Run paired-design statistical tests on the result.

> **Roadmap note.** Path B uses a `SyntheticOrb` shim purely so this notebook is runnable end-to-end without bundled paired data. Once a bundled paired demo dataset ships, the synthetic shim can be retired in favor of loading two real `Orb`s on Path A. *(TODO: drop `SyntheticOrb` when the demo dataset lands.)*

In [ ]:
from wizards_staff import Orb
from wizards_staff.drug_response import compare_baseline_dosing

---

# Section 11: When to Use Drug Response Analysis

**Learning objective**: Understand when paired analysis is appropriate and how to organize your data.

---

Drug response analysis is designed for **paired experimental designs** where:
- The **same sample** is imaged **twice** (before and after drug treatment)
- You want to measure **fold change** or **percent change** in activity
- You need to account for **sample-to-sample variability** by using each sample as its own control

### Common Use Cases

| Experimental Design | Example |
|--------------------|---------|
| Acute drug effects | Image organoid → apply TTX → image same organoid |
| Dose-response curves | Baseline → 1μM → 10μM → 100μM (sequential imaging) |
| Washout experiments | Baseline → drug → wash → measure recovery |
| Stimulation studies | Baseline → optogenetic/electrical stim → measure response |

### Key Advantages of Paired Analysis

1. **Increased statistical power** - Each sample serves as its own control
2. **Accounts for baseline variability** - Normalizes away sample-to-sample differences
3. **Biologically intuitive** - "2x increase in firing rate" is clearer than "treated group has mean of 15 vs control mean of 8"

---

# Section 12: Setting Up Paired Data

**Learning objective**: Organize your data for paired analysis.

---

## Data Organization

For drug response analysis, you'll have **two separate Lizard-Wizard runs** (recommended approach):

```
/path/to/baseline_results/     ← Imaging BEFORE drug treatment
├── Sample_A1_pre_dff-dat.npy
├── Sample_A1_pre_cnm-A.npy
├── Sample_A2_pre_dff-dat.npy
├── ...

/path/to/dosing_results/       ← Imaging AFTER drug treatment  
├── Sample_A1_post_dff-dat.npy
├── Sample_A1_post_cnm-A.npy
├── Sample_A2_post_dff-dat.npy
├── ...
```

### Metadata Files

The metadata files for both runs should have matching `Well` values to enable pairing:

**Baseline metadata:**
```csv
Sample,Well,Frate,Treatment
Sample_A1_pre,A1,30,Control
Sample_A2_pre,A2,30,Control
Sample_B1_pre,B1,30,DrugX
Sample_B2_pre,B2,30,DrugX
```

**Dosing metadata:**
```csv
Sample,Well,Frate,Treatment
Sample_A1_post,A1,30,Control
Sample_A2_post,A2,30,Control  
Sample_B1_post,B1,30,DrugX
Sample_B2_post,B2,30,DrugX
```

💡 **Key point**: The `Well` values must match between baseline and dosing for pairing to work!

---

# Section 13: Running the Drug Response Comparison

**Learning objective**: Execute the drug response analysis workflow.

---

The next two cells offer **two paths** to a pair of `Orb`s. Pick **one**:

- **Path A — Real paired data.** You have two Lizard-Wizard output folders. Edit the paths in the next cell, flip the guard, and run it.
- **Path B — Synthetic demo.** You have *one* analyzed `Orb` (from the main tutorial) and want to see the workflow without bundling a paired dataset. Run the Path B cell instead.

The cells after Section 13 are agnostic to which path you took.

## Path A: Real paired data

The cell below is **guarded** — it does nothing until you point `BASELINE_FOLDER` and `DOSING_FOLDER` at real folders that exist on disk. If they don't exist, the cell prints a hint and falls through to Path B.

In [ ]:
# =============================================================================
# PATH A: REAL PAIRED DATA
# =============================================================================
# Edit these four paths to point at your two Lizard-Wizard runs and their
# metadata CSVs. The cell only loads the Orbs if all four paths exist on
# disk, so it's safe to Run-All this notebook on a fresh checkout.
import os

BASELINE_FOLDER   = "/path/to/baseline_results"   # Pre-drug imaging
DOSING_FOLDER     = "/path/to/dosing_results"     # Post-drug imaging
METADATA_BASELINE = "/path/to/metadata_baseline.csv"
METADATA_DOSING   = "/path/to/metadata_dosing.csv"

orb_baseline = None
orb_dosing   = None

_paths_exist = all(
    os.path.exists(p)
    for p in (BASELINE_FOLDER, DOSING_FOLDER, METADATA_BASELINE, METADATA_DOSING)
)

if _paths_exist:
    print("Creating Orbs for baseline and dosing data...")
    orb_baseline = Orb(
        results_folder=BASELINE_FOLDER,
        metadata_file_path=METADATA_BASELINE,
    )
    orb_dosing = Orb(
        results_folder=DOSING_FOLDER,
        metadata_file_path=METADATA_DOSING,
    )
    print(f"Baseline: {orb_baseline.num_shards} samples loaded")
    print(f"Dosing:   {orb_dosing.num_shards} samples loaded")
else:
    print(
        "Path A skipped — at least one of BASELINE_FOLDER / DOSING_FOLDER /\n"
        "METADATA_BASELINE / METADATA_DOSING does not exist.\n"
        "Edit the paths above to use your real paired data, or run Path B\n"
        "below for the synthetic demo."
    )

## Path B: Synthetic demo

If Path A was skipped (or you just want a quick demo), this path fabricates **two** independent paired-data snapshots from a single already-analyzed `Orb` named `orb`. Use the `Orb` you built in the main tutorial — for example by running Part 1 of [`tutorial-notebook.ipynb`](tutorial-notebook.ipynb) up through `orb.run_all(...)` and then continuing here.

Two important properties of the cell below:

1. **No aliasing.** Both `orb_baseline` and `orb_dosing` are independent `SyntheticOrb` snapshots derived from `orb`. The user's real `orb` is **never** reassigned and its analysis state is **never** touched.
2. **No-op `run_all`.** `SyntheticOrb` carries pre-computed `frpm_data` / `rise_time_data` / `fwhm_data`, so its `run_all(...)` is a no-op. The next cell relies on this to keep Path A's analysis call uniform across both paths.

> **TODO:** retire `SyntheticOrb` once a bundled paired demo dataset ships — at that point Path A becomes the runnable default and this shim is no longer needed.

In [ ]:
# =============================================================================
# PATH B: SYNTHETIC PAIRED DATA (demo only)
# =============================================================================
# Builds TWO independent SyntheticOrb snapshots from a single analyzed
# `orb` — one as the baseline, one as the dosing. The real `orb` is not
# reassigned and its analysis state is not mutated.
import numpy as np


class SyntheticOrb:
    """
    Minimal orb-like object for demo'ing drug response analysis without
    bundled paired data. Carries snapshots of metadata, shard refs, and
    the three event-level metric DataFrames, optionally scaled by a
    random multiplier per row.

    `run_all` is a no-op: the snapshot already contains analyzed data.
    """

    def __init__(self, real_orb, multiplier_range=(1.0, 1.0), seed=42):
        np.random.seed(seed)
        self.metadata = real_orb.metadata.copy()
        self._shards = real_orb._shards  # only used for sample pairing

        for attr, value_col in (
            ("_frpm_data",      "Firing Rate Per Min"),
            ("_rise_time_data", "Rise Times"),
            ("_fwhm_data",      "FWHM Values"),
        ):
            src = getattr(real_orb, attr.lstrip("_"))
            if src is None:
                setattr(self, attr, None)
                continue
            df = src.copy()
            mults = np.random.uniform(*multiplier_range, size=len(df))
            df[value_col] = df[value_col] * mults
            setattr(self, attr, df)

    @property
    def frpm_data(self):
        return self._frpm_data

    @property
    def rise_time_data(self):
        return self._rise_time_data

    @property
    def fwhm_data(self):
        return self._fwhm_data

    def run_all(self, *args, **kwargs):
        """No-op: snapshots already contain analyzed data."""
        return None


_path_b_source = None
if orb_baseline is None or orb_dosing is None:
    try:
        _path_b_source = orb  # noqa: F821 — supplied by the user from the main tutorial
    except NameError:
        raise NameError(
            "Path B needs an analyzed `orb` in this notebook's namespace.\n"
            "Run Part 1 of the main tutorial (notebooks/tutorial-notebook.ipynb)\n"
            "up through `orb.run_all(...)`, then re-run this cell."
        )

    orb_baseline = SyntheticOrb(_path_b_source, multiplier_range=(1.0, 1.0), seed=1)
    orb_dosing   = SyntheticOrb(_path_b_source, multiplier_range=(0.5, 4.0), seed=42)
    print("Created synthetic paired Orbs (Path B).")
else:
    print("Path A already produced real Orbs — skipping Path B.")

## Run the standard analysis on both timepoints

For real `Orb`s (Path A) this runs the full pipeline (firing rate, rise time, FWHM, …) on each timepoint independently. For `SyntheticOrb`s (Path B) the snapshots already carry analyzed data, so `run_all` is a no-op and we skip it explicitly.

`frate` is read from `orb_baseline.metadata['Frate']` rather than hard-coded so the notebook works at whatever frame rate your data was acquired at.

In [ ]:
# Step 3: Run standard analysis on BOTH timepoints
# --------------------------------------------------
_synthetic = isinstance(orb_baseline, SyntheticOrb) or isinstance(orb_dosing, SyntheticOrb)

if _synthetic:
    print("Synthetic mode (Path B): skipping run_all — snapshots are already analyzed.")
else:
    # Pull frate from metadata rather than hard-coding it.
    frate = int(orb_baseline.metadata["Frate"].iloc[0])
    print(f"Running analysis at frate={frate} fps...")

    print("\nAnalyzing baseline samples...")
    orb_baseline.run_all(frate=frate, show_plots=False, save_files=False)

    print("\nAnalyzing dosing samples...")
    orb_dosing.run_all(frate=frate, show_plots=False, save_files=False)

    print("\n✅ Analysis complete for both timepoints!")

In [ ]:
# Step 4: Run drug response analysis
# ------------------------------------
# This pairs samples, computes fold changes, and creates visualizations
from wizards_staff import drug_response
from wizards_staff.drug_response import compare_baseline_dosing

results = compare_baseline_dosing(
    baseline_orb=orb_baseline,
    dosing_orb=orb_dosing,
    pair_by="Well",              # Column used to match baseline↔dosing samples
    metrics=["frpm", "rise_time", "fwhm"],  # Which metrics to analyze
    normalization="fold_change", # Options: "fold_change", "percent_change", "delta"
    group_col="Treatment",       # Optional: group results by this column
    aggregate=True,              # Aggregate neuron-level data to sample means
    agg_func="mean",             # Aggregation function
    time_unit="ms",              # Use milliseconds (auto-detects frame_rate from metadata)
    show_plots=True,             # Display visualizations
    save_files=True,             # Save CSVs and plots
    output_dir="./drug_response_outputs"
)

# View summary of paired samples
print("\nSample Pairing Summary:")
display(results["pairs"])


In [ ]:
# Step 5: Explore the results
# -----------------------------

# View firing rate fold changes
print("Firing Rate Drug Response:")
display(results["frpm"].head())

# View summary statistics by group
if "summary" in results:
    print("\nSummary Statistics by Treatment Group:")
    display(results["summary"])


---

# Section 14: Understanding Normalization Methods

**Learning objective**: Choose the right normalization for your analysis.

---

Drug response analysis offers several normalization options:

| Method | Formula | "No Change" Value | Best For |
|--------|---------|-------------------|----------|
| **fold_change** | dosing / baseline | 1.0 | Multiplicative effects (2x, 3x increase) |
| **percent_change** | ((dosing - baseline) / baseline) × 100 | 0% | Percentage effects (+50%, -25%) |
| **delta** | dosing - baseline | 0 | Absolute differences |
| **log2_fold_change** | log₂(dosing / baseline) | 0 | Symmetric up/down visualization |

### Interpreting Fold Change Values

| Fold Change | Interpretation |
|-------------|---------------|
| 2.0 | Activity **doubled** after drug treatment |
| 1.0 | **No change** - drug had no effect |
| 0.5 | Activity **halved** (50% reduction) |
| 0.0 | Complete **elimination** of activity |

In [ ]:
# Alternative: Using percent_change normalization
# -------------------------------------------------

results_pct = compare_baseline_dosing(
    baseline_orb=orb_baseline,
    dosing_orb=orb_dosing,
    pair_by="Well",
    metrics=["frpm"],
    normalization="percent_change",  # Changed from fold_change
    group_col="Treatment",
    show_plots=True,
    save_files=False
)

# A percent_change of +100% = doubled, -50% = halved
print("\nPercent Change Results:")
display(results_pct["frpm"][["pair_id", "baseline_Firing Rate Per Min", 
                              "dosing_Firing Rate Per Min", "percent_change"]])


---

# Section 15: Statistical Tests for Paired Data

**Learning objective**: Test whether drug effects are statistically significant.

---

After computing fold changes, you may want to test:

1. **One-sample test**: Is the fold change different from 1.0 (no change)?
2. **Two-sample test**: Is the fold change different between treatment groups?

In [ ]:
# Statistical tests on drug response data
# -----------------------------------------

from scipy import stats as sp_stats
import numpy as np

# Get fold change values
frpm_df = results["frpm"]

# TEST 1: One-sample t-test - Is overall fold change different from 1.0?
# ----------------------------------------------------------------------
fold_changes = frpm_df["fold_change"].dropna()
t_stat, p_value = sp_stats.ttest_1samp(fold_changes, popmean=1.0)

print("One-Sample T-Test: Is drug effect different from no change (fold=1.0)?")
print(f"   Mean fold change: {fold_changes.mean():.3f}")
print(f"   t-statistic: {t_stat:.3f}")
print(f"   p-value: {p_value:.4f}")
if p_value < 0.05:
    direction = "increased" if fold_changes.mean() > 1 else "decreased"
    print(f"   ✅ Significant! Drug treatment {direction} activity.")
else:
    print(f"   ⚠️ No significant drug effect detected.")

# TEST 2: Compare drug effect between treatment groups (if applicable)
# ---------------------------------------------------------------------
if "Treatment" in frpm_df.columns:
    groups = frpm_df["Treatment"].unique()
    if len(groups) == 2:
        group1_fc = frpm_df[frpm_df["Treatment"] == groups[0]]["fold_change"].dropna()
        group2_fc = frpm_df[frpm_df["Treatment"] == groups[1]]["fold_change"].dropna()
        
        t_stat2, p_value2 = sp_stats.ttest_ind(group1_fc, group2_fc)
        
        print(f"\nTwo-Sample T-Test: Comparing {groups[0]} vs {groups[1]}")
        print(f"   {groups[0]} mean fold change: {group1_fc.mean():.3f} (n={len(group1_fc)})")
        print(f"   {groups[1]} mean fold change: {group2_fc.mean():.3f} (n={len(group2_fc)})")
        print(f"   t-statistic: {t_stat2:.3f}")
        print(f"   p-value: {p_value2:.4f}")
        if p_value2 < 0.05:
            print(f"   ✅ Significant difference between groups!")
        else:
            print(f"   ⚠️ No significant difference between groups.")


---

# Section 16: Visualizing Drug Response

**Learning objective**: Create publication-ready visualizations for paired data.

---

The drug response analysis automatically generates three types of plots:

1. **Paired Line Plot** - Shows each sample's trajectory from baseline to dosing
2. **Fold Change Distribution** - Violin/box plot of normalized values with reference line
3. **Baseline vs Dosing Scatter** - X-Y plot with identity line (y=x)

## Custom Visualizations

In [ ]:
# Custom plotting with individual functions
# -------------------------------------------

from wizards_staff.plotting import (
    plot_paired_lines,
    plot_fold_change_distribution,
    plot_baseline_vs_dosing_scatter
)

# Get the FRPM results DataFrame
frpm_df = results["frpm"]

# Custom paired line plot with different colors
plot_paired_lines(
    data=frpm_df,
    metric="frpm",
    baseline_col="baseline_Firing Rate Per Min",
    dosing_col="dosing_Firing Rate Per Min",
    group_col="Treatment",
    title="Firing Rate: Before vs After Drug Treatment",
    palette="Dark2",          # Different color palette
    figsize=(10, 7),
    line_alpha=0.8,
    marker_size=100,
    show_plots=True,
    save_files=False
)


## Drug Response Analysis Summary

### Output Files Generated

When `save_files=True`, the following outputs are created:

| File | Description |
|------|-------------|
| `drug-response-frpm.csv` | Firing rate baseline, dosing, and fold changes |
| `drug-response-rise_time.csv` | Rise time baseline, dosing, and fold changes |
| `drug-response-fwhm.csv` | FWHM baseline, dosing, and fold changes |
| `drug-response-summary.csv` | Summary statistics per group |
| `drug_response_*_paired_lines.png` | Paired line plots |
| `drug_response_*_distribution.png` | Fold change distribution plots |
| `drug_response_*_scatter.png` | Baseline vs dosing scatter plots |
